# WienerNet — Results Analysis

Thin orchestration notebook for analysing one or more trained runs. All
heavy lifting (metrics computation, temporal aggregation, plotting helpers)
lives in the `wienernet.evaluation` package.

Typical workflow:
1. Train models from the terminal (`python scripts/train.py …`).
2. Run `python scripts/evaluate.py --run outputs/X/ outputs/Y/ …` to produce
   `evaluation_summary/{long,summary}.csv`.
3. Open this notebook to slice + plot whatever you need.


In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from wienernet.evaluation import standard_errors, cross_seed_pivot

sns.set_theme(context="notebook", style="whitegrid")
pd.set_option("display.max_rows", 200)


## 1. Load the cross-run summary

In [ ]:
# Edit this path to point at your evaluation_summary directory
SUMMARY_DIR = Path("../outputs/evaluation_summary")

long_df = pd.read_csv(SUMMARY_DIR / "long.csv")
print(f"long_df: {len(long_df)} rows, columns: {list(long_df.columns)}")
long_df.head()


## 2. Median-across-seeds summary

In [ ]:
summary = standard_errors(long_df, group_cols=["variant", "resolution", "target", "metric"])
# Slice to the headline numbers (raw resolution, primary targets)
focus = summary.query("resolution == 'raw' and target in ['nee', 'f', 'E0', 'rb']")
focus


## 3. Headline table: variant × metric (median across seeds)

In [ ]:
pivot_nee = cross_seed_pivot(long_df, target="nee", resolution="raw",
                              index="variant", columns="metric", aggfunc="median")
pivot_nee.round(4)


## 4. Temporal robustness: how do metrics change across resolutions?

In [ ]:
target = "nee"
metric = "mae"
df = long_df[(long_df.target == target) & (long_df.metric == metric)]

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(df, x="resolution", y="value", hue="variant",
            order=["raw", "daily", "weekly", "monthly", "quarterly"], ax=ax)
ax.set_title(f"{metric.upper()} on '{target}' across temporal resolutions")
ax.set_ylabel(metric)
plt.tight_layout()
plt.show()


## 5. Distributional metrics (MMD, KL, Wasserstein)

In [ ]:
dist_metrics = ["mmd", "kl", "wasserstein"]
dist = long_df[(long_df.target == "nee") &
               (long_df.metric.isin(dist_metrics)) &
               (long_df.resolution == "raw")]

fig, axes = plt.subplots(1, len(dist_metrics), figsize=(15, 4))
for ax, m in zip(axes, dist_metrics):
    sub = dist[dist.metric == m]
    sns.barplot(sub, x="variant", y="value", ax=ax)
    ax.set_title(f"{m.upper()} on NEE (lower = better)")
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()


## 6. Per-site breakdown (single run)

In [ ]:
# Pick a run directory to inspect site-level metrics
RUN_DIR = Path("../outputs").glob("*piae_sde_sampling*")
RUN_DIR = sorted(RUN_DIR)[-1]
print(f"Inspecting: {RUN_DIR}")

per_site = pd.read_csv(RUN_DIR / "metrics" / "per_site.csv")
per_site_focus = per_site.query("target == 'nee' and metric in ['mae', 'mmd', 'r2']")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, m in zip(axes, ["mae", "mmd", "r2"]):
    sub = per_site_focus[per_site_focus.metric == m]
    sns.barplot(sub, x="site", y="value", ax=ax)
    ax.set_title(f"NEE {m.upper()} per site")
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()


## 7. Time-series plot of predictions vs ground truth

In [ ]:
# predictions.parquet lives next to each run's metrics/
preds = pd.read_parquet(RUN_DIR / "metrics" / "predictions.parquet")
print(f"predictions: {len(preds)} rows, columns: {list(preds.columns)[:12]}...")
preds.head(3)


In [ ]:
# Plot a slice (one site, one month)
site = preds["site"].iloc[0] if "site" in preds.columns else None
slice_df = preds.copy()
if site is not None:
    slice_df = slice_df[slice_df["site"] == site]
slice_df = slice_df.iloc[:480]   # first ~10 days at 30-min cadence

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(slice_df["DateTime"], slice_df["gt_nee"], label="NEE (observed)", lw=1.5)
ax.plot(slice_df["DateTime"], slice_df["pred_nee"], label="NEE (predicted)", lw=1.2, alpha=0.85)
ax.set_title(f"NEE predictions — {site}")
ax.set_xlabel("Time")
ax.set_ylabel("NEE (µmol m⁻² s⁻¹)")
ax.legend()
plt.tight_layout()
plt.show()
